# Refuerzo: Clustering Aplicado y Despliegue de Soluciones ML
## Ingeniería del Conocimiento (ISO56B) — UNCP
### Msc. Jaime Antonio Huaytalla Pariona — Semestre 2026-II

**Objetivo:** Aplicar algoritmos de clustering a un dataset real, comparar resultados, e introducir el despliegue de modelos ML con serialización y APIs.

**Nivel:** Intermedio → Avanzado

**Contenido:**
1. Clustering con dataset Wine
2. Comparación completa de algoritmos
3. Serialización de modelos (joblib)
4. API REST con Flask
5. Interfaz con Gradio


## 1. Configuración y Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Cargar Wine dataset
wine = load_wine()
X = wine.data
y_true = wine.target  # Solo para validación, NO se usa en clustering
feature_names = wine.feature_names

df = pd.DataFrame(X, columns=feature_names)
df['true_class'] = y_true

print(f"Dataset Wine: {X.shape[0]} muestras × {X.shape[1]} features")
print(f"Clases reales (solo para validar): {np.bincount(y_true)}")
print(f"\nFeatures: {list(feature_names)}")


In [ ]:
# Exploración rápida
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribuciones de features clave
for ax, feat in zip(axes.ravel(), ['alcohol', 'malic_acid', 'flavanoids', 'color_intensity']):
    for cls in range(3):
        mask = y_true == cls
        ax.hist(df[feat][mask], bins=15, alpha=0.5, label=f'Clase {cls}')
    ax.set_xlabel(feat)
    ax.set_ylabel('Frecuencia')
    ax.set_title(f'Distribución: {feat}')
    ax.legend()

plt.suptitle('Distribución de Features por Clase Real (Wine)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observación: las clases se separan en varios features.")
print("¿Podrá el clustering encontrar estos 3 grupos sin usar las etiquetas?")


## 2. Aplicar Clustering al Dataset Wine

### 2.1 Estandarización (obligatoria)


In [ ]:
# Estandarizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Antes de estandarizar:")
print(f"  Medias:  {X.mean(axis=0)[:4].round(2)} ...")
print(f"  Stds:    {X.std(axis=0)[:4].round(2)} ...")
print(f"\nDespués de estandarizar:")
print(f"  Medias:  {X_scaled.mean(axis=0)[:4].round(4)} ...")
print(f"  Stds:    {X_scaled.std(axis=0)[:4].round(4)} ...")


### 2.2 Selección de K

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Coeficiente de Silueta')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Selección de K para Wine Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(silhouettes)]
print(f"Mejor K según Silhouette: {best_k} (score = {max(silhouettes):.3f})")
print(f"Nota: sabemos que hay 3 clases reales. Veamos si el clustering lo encuentra.")


### 2.3 Comparación de Algoritmos

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, adjusted_rand_score,
                             normalized_mutual_info_score)

# Definir algoritmos
algoritmos = {
    'K-Means (K=3)':      KMeans(n_clusters=3, random_state=42),
    'Jerárquico (K=3)':   AgglomerativeClustering(n_clusters=3, linkage='ward'),
    'GMM (K=3)':          GaussianMixture(n_components=3, random_state=42),
    'DBSCAN (eps=3)':     DBSCAN(eps=3, min_samples=5),
}

results = []
labels_dict = {}

for nombre, algo in algoritmos.items():
    if hasattr(algo, 'fit_predict'):
        labels = algo.fit_predict(X_scaled)
    else:
        algo.fit(X_scaled)
        labels = algo.predict(X_scaled)

    labels_dict[nombre] = labels

    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    n_noise = sum(labels == -1)

    if n_clusters > 1 and sum(mask) > n_clusters:
        sil = silhouette_score(X_scaled[mask], labels[mask])
        ch = calinski_harabasz_score(X_scaled[mask], labels[mask])
        db = davies_bouldin_score(X_scaled[mask], labels[mask])
        ari = adjusted_rand_score(y_true[mask], labels[mask])
        nmi = normalized_mutual_info_score(y_true[mask], labels[mask])
    else:
        sil = ch = ari = nmi = 0
        db = 999

    results.append({
        'Algoritmo': nombre, 'Clusters': n_clusters, 'Ruido': n_noise,
        'Silhouette': sil, 'ARI': ari, 'NMI': nmi
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

print("\nARI y NMI comparan contra las clases reales:")
print("  1.0 = coincidencia perfecta, 0.0 = asignación aleatoria")


In [ ]:
# Visualizar resultados con PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (nombre, labels) in zip(axes, labels_dict.items()):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', s=25, alpha=0.7)
    ax.set_title(nombre, fontsize=11)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('Resultados de Clustering (Wine, proyección PCA)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Despliegue de Modelos ML

Una vez que tenemos un buen modelo, necesitamos **ponerlo en producción** para que otros puedan usarlo.

### 3.1 Serialización: Guardar el Modelo

Usamos `joblib` para guardar el pipeline completo (preprocesamiento + modelo).


In [ ]:
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Ejemplo: entrenar un clasificador para desplegar
# (usamos las etiquetas reales para este ejemplo de clasificación)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_true, test_size=0.2, random_state=42, stratify=y_true
)

# Pipeline completo
pipeline = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=100, random_state=42)
)
pipeline.fit(X_train, y_train)

accuracy = pipeline.score(X_test, y_test)
print(f"Accuracy del modelo: {accuracy:.4f}")

# Guardar el pipeline completo
joblib.dump(pipeline, 'modelo_wine.joblib')
print("\n✅ Modelo guardado como 'modelo_wine.joblib'")

# Verificar: cargar y predecir
modelo_cargado = joblib.load('modelo_wine.joblib')
pred = modelo_cargado.predict(X_test[:3])
print(f"\nPredicciones de prueba: {pred}")
print(f"Valores reales:         {y_test[:3]}")
print(f"\n→ El pipeline incluye el scaler + modelo. Al predecir, estandariza automáticamente.")


### 3.2 API REST con Flask

Flask permite crear un servicio web que recibe datos y devuelve predicciones.

**Nota:** Este código se ejecutaría en un archivo `.py` separado, no en Colab. Aquí lo mostramos para entender la estructura.


In [ ]:
# === Código para app.py (NO ejecutar en Colab) ===
flask_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
modelo = joblib.load('modelo_wine.joblib')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        datos = request.get_json()
        features = np.array(datos['features']).reshape(1, -1)
        prediccion = modelo.predict(features)
        proba = modelo.predict_proba(features)
        return jsonify({
            'clase': int(prediccion[0]),
            'probabilidades': proba[0].tolist(),
            'status': 'ok'
        })
    except Exception as e:
        return jsonify({'error': str(e), 'status': 'error'}), 400

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy'})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

print("=== app.py (Flask API) ===")
print(flask_code)

print("\nPara ejecutar:")
print("  1. Guardar como app.py")
print("  2. En terminal: python app.py")
print("  3. Probar: curl -X POST http://localhost:5000/predict \\")
print("       -H 'Content-Type: application/json' \\")
print("       -d '{"features": [13.2, 1.78, 2.14, 11.2, 100, 2.65, 2.76, 0.26, 1.28, 4.38, 1.05, 3.40, 1050]}'")


### 3.3 API REST con FastAPI (alternativa moderna)

FastAPI genera documentación automática y es más rápido.


In [ ]:
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI(title="Wine Classifier API")
modelo = joblib.load('modelo_wine.joblib')

class WineInput(BaseModel):
    features: list[float]

class WinePrediction(BaseModel):
    clase: int
    nombre_clase: str
    probabilidades: list[float]

@app.post("/predict", response_model=WinePrediction)
def predict(data: WineInput):
    X = np.array(data.features).reshape(1, -1)
    pred = modelo.predict(X)[0]
    proba = modelo.predict_proba(X)[0]
    nombres = ["class_0", "class_1", "class_2"]
    return WinePrediction(
        clase=int(pred),
        nombre_clase=nombres[pred],
        probabilidades=proba.tolist()
    )

@app.get("/health")
def health():
    return {"status": "healthy"}
'''

print("=== main.py (FastAPI) ===")
print(fastapi_code)
print("\nPara ejecutar:")
print("  1. pip install fastapi uvicorn")
print("  2. uvicorn main:app --reload")
print("  3. Abrir http://localhost:8000/docs → documentación interactiva")


### 3.4 Interfaz con Gradio (demos rápidas)

Gradio crea interfaces web interactivas con pocas líneas de código.


In [ ]:
# Instalar Gradio
!pip install -q gradio


In [ ]:
import gradio as gr

# Cargar modelo
modelo = joblib.load('modelo_wine.joblib')
nombres_clase = ['Clase 0 (Barolo)', 'Clase 1 (Grignolino)', 'Clase 2 (Barbera)']

def clasificar_vino(alcohol, malic_acid, ash, alcalinity, magnesium,
                     total_phenols, flavanoids, nonflavanoid, proanthocyanins,
                     color_intensity, hue, od280, proline):
    features = np.array([[alcohol, malic_acid, ash, alcalinity, magnesium,
                          total_phenols, flavanoids, nonflavanoid, proanthocyanins,
                          color_intensity, hue, od280, proline]])
    pred = modelo.predict(features)[0]
    proba = modelo.predict_proba(features)[0]

    resultado = {nombres_clase[i]: float(proba[i]) for i in range(3)}
    return resultado

# Crear interfaz
demo = gr.Interface(
    fn=clasificar_vino,
    inputs=[
        gr.Number(label="Alcohol", value=13.0),
        gr.Number(label="Malic Acid", value=1.7),
        gr.Number(label="Ash", value=2.4),
        gr.Number(label="Alcalinity of Ash", value=15.6),
        gr.Number(label="Magnesium", value=127.0),
        gr.Number(label="Total Phenols", value=2.8),
        gr.Number(label="Flavanoids", value=3.0),
        gr.Number(label="Nonflavanoid Phenols", value=0.3),
        gr.Number(label="Proanthocyanins", value=2.3),
        gr.Number(label="Color Intensity", value=5.6),
        gr.Number(label="Hue", value=1.0),
        gr.Number(label="OD280/OD315", value=3.5),
        gr.Number(label="Proline", value=1500),
    ],
    outputs=gr.Label(num_top_classes=3, label="Clasificación"),
    title="🍷 Clasificador de Vinos",
    description="Ingrese las características del vino para clasificarlo.",
)

# Lanzar (en Colab se abre automáticamente)
demo.launch(share=True)


## 4. Resumen Final

### Clustering
| Algoritmo | Fortaleza | Debilidad |
|-----------|----------|-----------|
| K-Means | Rápido, simple | Asume formas esféricas, requiere K |
| DBSCAN | Cualquier forma, detecta ruido | Sensible a eps |
| Jerárquico | Dendrograma visual, no requiere K fijo | Lento para datos grandes |
| GMM | Modela incertidumbre (probabilidades) | Asume distribuciones gaussianas |

### Despliegue
| Herramienta | Uso |
|-------------|-----|
| **joblib** | Guardar/cargar modelos (serialización) |
| **Flask** | API REST clásica y confiable |
| **FastAPI** | API REST moderna con documentación automática |
| **Gradio** | Demos y prototipos interactivos rápidos |

### Flujo completo:
Datos → Preprocesamiento → Clustering/Modelado → Evaluación → **Serialización → Despliegue → Predicción en producción**

---
*Material de refuerzo — Ingeniería del Conocimiento (ISO56B) — UNCP*
